### **Ollama Setup**

*This gets the Ollama server running and pulls the models. We will use `llama3` for complex reasoning/QA and `gemma` to show a lightweight alternative for summarization.*

In [ ]:
# Environment Setup & Server Start

!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log

nohup: ignoring input and redirecting stderr to stdout


In [ ]:
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
print("Ollama server should be ready on http://localhost:11434")

Ollama server should be ready on http://localhost:11434


In [ ]:
# Pull Models
!ollama pull llama3

In [ ]:
!ollama pull gemma

In [ ]:
# Standard call_ollama Helper

import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

In [ ]:
def call_ollama(prompt, model="llama3", temperature=0.7, stream=False):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": {
            "temperature": temperature
        }
    }

    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")

    if stream:
        for line in resp.text.splitlines():
            if not line.strip(): continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
        print("\n")
    else:
        return resp.json().get("response", "")

### **1: Text Generation & Classification**

**Concept:** Using LLMs for Zero-Shot Classification. We will fetch a real, raw news article from Scikit-Learn's public "20 Newsgroups" dataset and ask the model to classify its intent/topic without any training data.

In [ ]:
# Fetch Real Text Data for Classification

from sklearn.datasets import fetch_20newsgroups

In [ ]:
# Fetch a real dataset of newsgroup posts (we'll grab the 'sci.space' and 'rec.autos' categories)
categories = ['sci.space', 'rec.autos', 'comp.graphics']
real_data = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

In [ ]:
# Select one random real text snippet from the dataset
sample_text = real_data.data[42].strip()

In [ ]:
print(" Real Raw Text Snippet ")
print(sample_text[:500] + "...\n")

 Real Raw Text Snippet 
From another space forum
    When workers at the Kennedy Space Center disassembled the STS-56
 solid rocket boosters they were surprised to find a pair of pliers
 lodged into the outside base of the right hand SRB.  The tool survived
 the trip from the launch pad up to approximately a 250,000 foot
 altitude, then down to splashdown and towing back to KSC.

    NASA spokesperson Lisa Malone told the media,

    "It's been a long time since something like this happened.  We've
 lost washers and bo...



In [ ]:
# Effect of Temperature on creativity vs. determinism
base_prompt = """
Write a product launch email for 'DataFlow Gen', an AI analytics assistant.
Tone: Urgent and professional.
Constraint: Keep it strictly under 50 words. Do not use the word 'revolutionary'.
"""

In [ ]:
print("--- Low Temperature (0.2) - Highly Deterministic ---")
print(call_ollama(base_prompt, model="llama3", temperature=0.2))

--- Low Temperature (0.2) - Highly Deterministic ---
Subject: Introducing DataFlow Gen - Unlock Your Data's Full Potential

Dear [Recipient],

We're thrilled to announce the launch of DataFlow Gen, an AI-powered analytics assistant that streamlines data insights and accelerates decision-making. Don't miss this opportunity to transform your organization's data strategy. Learn more and get started today!

Best, [Your Name]


In [ ]:
print("\n--- High Temperature (0.9) - Highly Creative ---")
print(call_ollama(base_prompt, model="llama3", temperature=0.9))


--- High Temperature (0.9) - Highly Creative ---
Subject: Introducing DataFlow Gen - Unlock Your Business Insights Today!

Dear [Recipient],

We're excited to introduce DataFlow Gen, an AI-powered analytics assistant designed to accelerate your business decisions. With its unparalleled speed and accuracy, you'll gain instant access to valuable insights. Learn more and get started today!


In [ ]:
# Zero-Shot Classification using Llama 3

classification_prompt = f"""
You are an expert text classifier. Read the following text snippet and classify it into exactly ONE of the following categories:
- Space & Astronomy
- Automobiles & Cars
- Computer Graphics
- Unknown

Text Snippet:
"{sample_text}"

Task:
1. Provide the Category Name.
2. Provide a 1-sentence explanation of why you chose this category based on keywords in the text.
"""

In [ ]:
print(" Classification Result ")
result = call_ollama(prompt=classification_prompt, model="llama3", stream=False)
print(result)

 Classification Result 
Category: Space & Astronomy

Explanation: I chose this category because the text snippet mentions specific terms related to space exploration, such as "Kennedy Space Center", "STS-56", "solid rocket boosters", and "Shuttle", indicating that it is a story about a space-related event or incident.


In [ ]:
# Few-Shot Prompting Pattern
few_shot_prompt = f"""
You are an expert intent classifier. Classify the text into exactly ONE of these categories:
[Space & Astronomy, Automobiles & Cars, Computer Graphics, Unknown]

Examples:
Text: "The new orbital telescope captured images of the nebula."
Label: Space & Astronomy

Text: "How do I fix the transmission on a 1998 Honda Civic?"
Label: Automobiles & Cars

Text: "{sample_text}"
Label:
"""

In [ ]:
print("--- Few-Shot Classification Result ---")
print(call_ollama(prompt=few_shot_prompt, model="llama3", temperature=0.1))

--- Few-Shot Classification Result ---
Label: Space & Astronomy


### **2: Document Summarization**

**Concept:** Handling long-context documents and abstractive summarization. We will use the `wikipedia` Python package to pull a massive, real-world article and use `gemma` (a highly efficient model for text tasks) to summarize it.

In [ ]:
# Fetch a Real, Lengthy Document

!pip install wikipedia-api -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 9.4 MB/s eta 0:00:00


In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia("GenAITraining/1.0", "en")
page = wiki.page("Generative_artificial_intelligence")
long_document = page.text[:8000] # Grabbing a large chunk

In [ ]:
# 1. Chunking the document
chunk_size = 4000
chunks = [long_document[i:i+chunk_size] for i in range(0, len(long_document), chunk_size)]

print(f"Split document into {len(chunks)} chunks.\n")

Split document into 2 chunks.



In [ ]:
# 2. MAP STEP: Summarize each chunk
chunk_summaries = []
for i, chunk in enumerate(chunks):
    map_prompt = f"Summarize the following text in exactly 2 bullet points:\n{chunk}"
    summary = call_ollama(map_prompt, model="gemma", temperature=0.3)
    chunk_summaries.append(summary)
    print(f"Chunk {i+1} Summary Generated.")

Chunk 1 Summary Generated.
Chunk 2 Summary Generated.


In [ ]:
# 3. REDUCE STEP: Final Synthesis
combined_summaries = "\n".join(chunk_summaries)
reduce_prompt = f"""
You are a senior analyst. Synthesize the following section summaries into a single, cohesive 3-sentence executive brief.
Summaries:
{combined_summaries}
"""

print("\n--- Final Map-Reduce Executive Brief ---")
print(call_ollama(reduce_prompt, model="llama3", temperature=0.4))


--- Final Map-Reduce Executive Brief ---
Here is a 3-sentence executive brief synthesizing the section summaries:

Generative Artificial Intelligence (AI) has emerged as a powerful tool for generating data in various forms, including text, images, videos, audio, and software code. This technology has become increasingly prevalent across industries such as software development, healthcare, entertainment, art, finance, media, and robotics, with applications like DALL-E and Midjourney revolutionizing content creation by enabling the generation of multimedia outputs from textual prompts. As a result, generative AI is poised to transform various sectors, offering innovative solutions for data generation, content creation, and process automation.


### **3: End-to-End Case Study - Resume Screening & QA**

**Concept:** Question Answering over specific contexts. We will download a real Markdown-formatted sample developer resume directly from a public open-source GitHub repository and build an AI recruiter screening tool.

In [ ]:
# Hardcoded Resume Markdown

resume_text = """
# Alex Mercer
**Senior Software Engineer**
Location: San Francisco, CA | Email: alex.mercer@example.com

## Summary
Back-end engineer with 6+ years of experience designing scalable microservices and cloud infrastructure.

## Experience
**TechCorp Solutions** | Senior Backend Engineer | Jan 2021 - Present
- Developed and deployed scalable microservices using Python, FastAPI, and Go.
- Led the migration of legacy monolith architectures to Docker and Kubernetes.
- Reduced database query latency by 40% through indexing and Redis caching.

**DataFlow Inc.** | Software Engineer | Jun 2018 - Dec 2020
- Built highly available REST APIs in Node.js and maintained legacy Python Django apps.
- Wrote unit and integration tests achieving 90% code coverage.

## Education
**B.S. in Computer Science**
University of California, Berkeley (2014 - 2018)

## Skills
- **Languages:** Python, Go, JavaScript, SQL
- **Tools & Cloud:** Docker, Kubernetes, AWS (EC2, S3, RDS), Git
"""

In [ ]:
print("--- Candidate Resume Loaded ---")
print(resume_text[:400] + "...\n")

--- Candidate Resume Loaded ---

# Alex Mercer
**Senior Software Engineer**
Location: San Francisco, CA | Email: alex.mercer@example.com

## Summary
Back-end engineer with 6+ years of experience designing scalable microservices and cloud infrastructure.

## Experience
**TechCorp Solutions** | Senior Backend Engineer | Jan 2021 - Present
- Developed and deployed scalable microservices using Python, FastAPI, and Go.
- Led the migr...



In [ ]:
# Resume Screening QA Workflow

qa_system_prompt = """
You are an HR compliance assistant. Answer questions based ONLY on the provided context.
If the answer is not explicitly stated in the context, you must reply strictly with: "Not found in provided documents."
Do not guess or infer. Provide a direct quote to support your answer if found.

Context:
{resume_text}

Question: {question}
"""

In [ ]:
print("--- Test 1: Grounded Factual Question ---")
q1 = qa_system_prompt.format(resume_text=resume_text, question="Does the candidate know Python?")
print(call_ollama(q1, model="llama3", temperature=0.1))

--- Test 1: Grounded Factual Question ---
Yes, according to the provided context:

**Summary**
Back-end engineer with 6+ years of experience designing scalable microservices and cloud infrastructure.

**Experience**
... Developed and deployed scalable microservices using **Python**, FastAPI, and Go.

**Skills**
- **Languages:** Python, Go, JavaScript, SQL


In [ ]:
print("\n--- Test 2: AI Safety & Refusal Test (Hallucination Check) ---")
q2 = qa_system_prompt.format(resume_text=resume_text, question="What is the candidate's age and marital status?")
print(call_ollama(q2, model="llama3", temperature=0.1))


--- Test 2: AI Safety & Refusal Test (Hallucination Check) ---
Not found in provided documents.


In [ ]:
# Resume Screening QA Workflow
qa_prompt = f"""

You are a Senior Technical Recruiter. Review the following candidate resume:



RESUME START

{resume_text}

RESUME END



Answer the following screening questions based ONLY on the provided resume:

1. **Years of Experience:** What is the total span of professional experience listed?

2. **Tech Stack Verification:** Does this candidate have explicit experience with Python or Go? Quote the specific line if yes.

3. **Red Flags / Gaps:** Are there any noticeable employment gaps or lacking educational requirements for a Senior Software Engineer role?

4. **Final Recommendation:** (Proceed to Interview / Reject)



Format your response cleanly with markdown headers for each question.

"""

In [ ]:
print(" AI Recruiter Screening Assessment ")

screening_report = call_ollama(prompt=qa_prompt, model="llama3", stream=True)

 AI Recruiter Screening Assessment 
**Screening Questions**

### 1. Years of Experience

The total span of professional experience listed is approximately 6+ years, spanning from 2018 to present.

### 2. Tech Stack Verification

Yes, this candidate has explicit experience with Python and Go. Specifically:

* **TechCorp Solutions**: Developed and deployed scalable microservices using Python, FastAPI, and Go.
* **DataFlow Inc.**: Built highly available REST APIs in Node.js (not explicitly listed as a strength) but also maintained legacy Python Django apps.

### 3. Red Flags / Gaps

There are no noticeable employment gaps or lacking educational requirements for a Senior Software Engineer role.

### 4. Final Recommendation

**Proceed to Interview**

Based on the provided resume, Alex Mercer appears to have the necessary technical skills and experience to excel as a Senior Software Engineer. The highlighted achievements in scalability, cloud infrastructure, and testing suggest that he has a